## Library

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Step 1 — Load Master Dataset

In [4]:
BASE_DIR = Path("/home/dawwi/Dicoding/Capstone/AI-Model")
DATA_PATH = BASE_DIR / "clean_dataset" / "master.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values(["city", "timestamp"]).reset_index(drop=True)

# Basic sanity checks
required_cols = {"city", "timestamp", "ghi", "temp_c", "clear_ghi"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in master.csv: {missing}")

### Step 2 — Buat Dataset Harian (target: GHI besok)

In [5]:
df["date"] = df["timestamp"].dt.date

daily = (
    df.groupby(["city", "date"])
    .agg(
        ghi_day=("ghi", "sum"),
        temp_mean=("temp_c", "mean"),
        clear_day=("clear_ghi", "sum"),
    )
    .reset_index()
)

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values(["city", "date"]).reset_index(drop=True)

### Step 3 — Feature Engineering

In [6]:
daily["month"] = daily["date"].dt.month.astype(np.int16)
daily["dayofyear"] = daily["date"].dt.dayofyear.astype(np.int16)

# Sky ratio: makin kecil = makin berawan
# Handle clear_day=0 (avoid division by zero)
daily["clear_day_safe"] = daily["clear_day"].replace(0, np.nan)
daily["sky_ratio"] = (daily["ghi_day"] / daily["clear_day_safe"]).clip(0, 1)
daily["sky_ratio"] = daily["sky_ratio"].fillna(0.0)  # jika clear_day=0, anggap sangat berawan

# Lag features per kota
daily = daily.sort_values(["city", "date"]).reset_index(drop=True)
for lag in [1, 2, 3, 7]:
    daily[f"ghi_lag_{lag}"] = daily.groupby("city")["ghi_day"].shift(lag)
    daily[f"sky_lag_{lag}"] = daily.groupby("city")["sky_ratio"].shift(lag)

# Rolling mean 7 hari (pakai histori, jadi shift(1) dulu)
daily["ghi_roll_7"] = (
    daily.groupby("city")["ghi_day"]
    .shift(1)
    .rolling(7)
    .mean()
    .reset_index(level=0, drop=True)
)

# Target: besok
daily["target_ghi_tomorrow"] = daily.groupby("city")["ghi_day"].shift(-1)

# Drop rows yang belum lengkap karena lag/rolling/target
daily = daily.dropna().reset_index(drop=True)

# Optional: buang kolom helper
daily = daily.drop(columns=["clear_day_safe"])

### Step 4 — Split Train/Test Time-series

In [7]:
CUT = pd.Timestamp("2024-01-01")

train = daily[daily["date"] < CUT].copy()
test = daily[daily["date"] >= CUT].copy()

if train.empty or test.empty:
    raise ValueError(
        f"Train/Test kosong. Cek rentang tanggal dataset. "
        f"Min date={daily['date'].min()}, Max date={daily['date'].max()}"
    )

### Step 5 — Baseline

In [8]:
test["pred_baseline"] = test["ghi_lag_1"]  # besok ≈ hari ini
mae_baseline = mean_absolute_error(test["target_ghi_tomorrow"], test["pred_baseline"])
print(f"MAE baseline (lag-1): {mae_baseline:,.4f}")

MAE baseline (lag-1): 812.2883


### Step 6 — Train Model (Gradient Boosting)

In [10]:
FEATURES = [
    "temp_mean", "sky_ratio", "month", "dayofyear",
    "ghi_lag_1", "ghi_lag_2", "ghi_lag_3", "ghi_lag_7",
    "sky_lag_1", "sky_lag_2", "sky_lag_3", "sky_lag_7",
    "ghi_roll_7",
]

X_train = train[FEATURES]
y_train = train["target_ghi_tomorrow"]

X_test = test[FEATURES]
y_test = test["target_ghi_tomorrow"]

model = HistGradientBoostingRegressor(
    random_state=42,
    max_depth=None,          # biarkan model cari struktur
    learning_rate=0.05,
    max_iter=500,
    l2_regularization=0.0,
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"MAE model:  {mae:,.4f}")
print(f"RMSE model: {rmse:,.4f}")
print(f"R2 model:   {r2:,.4f}")

# Per-city MAE (biar keliatan kota mana yang susah)
per_city = (
    test.assign(pred=pred)
    .groupby("city")
    .apply(lambda g: mean_absolute_error(g["target_ghi_tomorrow"], g["pred"]))
    .sort_values()
)
print("\nMAE per city:")
print(per_city.to_string())

# Save artifact
OUT_DIR = BASE_DIR / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

dump(
    {"model": model, "features": FEATURES, "cutoff": str(CUT.date())},
    OUT_DIR / "ghi_hgb.joblib",
)

print(f"\nSaved model to: {OUT_DIR / 'ghi_hgb.joblib'}")

MAE model:  623.3363
RMSE model: 868.8496
R2 model:   0.3348

MAE per city:
city
Bojonegoro     508.406626
Yogyakarta     508.776759
Kediri         531.931666
Surabaya       533.135368
Tasik          606.256021
Probolinggo    608.102865
Situbondo      609.331532
Jembrana       609.780178
Banyu Wangi    611.637937
Gerokgak       612.147854
Purworkorto    613.116657
Denpasar       613.383629
Buleleng       614.539817
Kubu           614.780482
Bandung        631.241708
Malang         639.485956
Jember         648.495562
Candipuro      651.771612
Jakarta        677.255419
cerbon         708.472522
Pekalongan     711.874370
Jepara         735.665540
Semarang       737.144351

Saved model to: /home/dawwi/Dicoding/Capstone/AI-Model/models/ghi_hgb.joblib
